### Grib file inspection

Inspect the .GRIB file downloaded from Copernicus Climate Platform.

In [33]:
import cfgrib

def inspect_grib_fast(grib_path):
    """
    Quickly list all variables and their metadata from a GRIB file.
    Uses cfgrib.open_datasets which returns lazy datasets – very fast.
    """
    print(f"GRIB file: {grib_path}\n")
    datasets = cfgrib.open_datasets(grib_path, backend_kwargs={'errors': 'ignore'})
    print(f"Number of datasets: {len(datasets)}")
    for i, ds in enumerate(datasets):
        var_names = list(ds.data_vars)
        dims = dict(ds.dims)
        print(f"\nDataset {i}:")
        print(f"  Variables: {var_names}")
        print(f"  Dimensions: {dims}")
        if 'step' in ds.coords:
            step_size = ds.step.size if hasattr(ds.step, 'size') else 1
            print(f"  'step' coordinate size: {step_size}")
        for var in var_names:
            shape = ds[var].shape
            print(f"    {var}: shape={shape}")
    return

In [34]:
# Example usage
grib_file = "/mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib"
inspect_grib_fast(grib_file)

GRIB file: /mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib



Ignoring index file '/mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '/mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib.5b7b6.idx' incompatible with GRIB file


Number of datasets: 2

Dataset 0:
  Variables: ['t2m']
  Dimensions: {'time': 8784, 'latitude': 721, 'longitude': 1440}
  'step' coordinate size: 1
    t2m: shape=(8784, 721, 1440)

Dataset 1:
  Variables: ['tp']
  Dimensions: {'time': 733, 'step': 12, 'latitude': 721, 'longitude': 1440}
  'step' coordinate size: 12
    tp: shape=(733, 12, 721, 1440)


/tmp/ipykernel_19202/2486217526.py:13: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  dims = dict(ds.dims)


### Clip Dataset to study area

In [15]:
import subprocess
import tempfile
import os
import geopandas as gpd
import xarray as xr

def clip_grib_with_cdo(grib_path, shapefile_path, output_path,
                       buffer_deg=0.5, resample_daily=True,
                       variables=None, verbose=True):
    """
    Clip a GRIB file to the bounding box of a shapefile using CDO.
    Automatically converts longitude to 0‑360 if the GRIB uses that range.
    """
    # ---- 1. Get bounding box from shapefile ----
    gdf = gpd.read_file(shapefile_path).to_crs("EPSG:4326")
    bbox = gdf.total_bounds
    lon_min = bbox[0] - buffer_deg
    lon_max = bbox[2] + buffer_deg
    lat_min = bbox[1] - buffer_deg
    lat_max = bbox[3] + buffer_deg

    # Clamp to valid ranges
    lon_min = max(-180, min(180, lon_min))
    lon_max = max(-180, min(180, lon_max))
    lat_min = max(-90, min(90, lat_min))
    lat_max = max(-90, min(90, lat_max))
    if verbose:
        print(f"Original bounding box (lon -180..180): [{lon_min:.4f}, {lon_max:.4f}], lat [{lat_min:.4f}, {lat_max:.4f}]")

    # Convert longitude to 0‑360 if the GRIB likely uses that
    # Many ERA5 GRIBs use longitudes 0..360. We'll try both.
    lon_min_360 = lon_min if lon_min >= 0 else lon_min + 360
    lon_max_360 = lon_max if lon_max >= 0 else lon_max + 360
    if verbose:
        print(f"Converted to 0‑360: [{lon_min_360:.4f}, {lon_max_360:.4f}]")

    # ---- 2. Create a temporary file for the subset ----
    with tempfile.NamedTemporaryFile(suffix='.nc', delete=False) as tmp:
        tmp_path = tmp.name

    # Try with 0‑360 first, then fallback to -180..180
    for lon_range in [(lon_min_360, lon_max_360), (lon_min, lon_max)]:
        lon_min_box, lon_max_box = lon_range
        if lon_min_box > lon_max_box:
            lon_min_box, lon_max_box = lon_max_box, lon_min_box

        cdo_cmd = [
            'cdo', '-f', 'nc', 'sellonlatbox',
            f'{lon_min_box},{lon_max_box},{lat_min},{lat_max}',
            grib_path, tmp_path
        ]
        if verbose:
            print(f"\nTrying: {' '.join(cdo_cmd)}")

        try:
            result = subprocess.run(cdo_cmd, capture_output=True, text=True, check=True)
            if verbose:
                print("CDO succeeded.")
            break
        except subprocess.CalledProcessError as e:
            if verbose:
                print("CDO failed with stderr:")
                print(e.stderr)
            if "no grid cells selected" in e.stderr:
                if verbose:
                    print("No cells selected with this longitude range, trying the other.")
                continue
            else:
                # Some other error – raise
                raise
    else:
        raise RuntimeError("Could not clip with either longitude range. Try increasing buffer_deg.")

    # ---- 3. Open the clipped dataset with xarray ----
    ds = xr.open_dataset(tmp_path, chunks={})

    # ---- 4. Select variables if requested ----
    if variables is not None:
        existing_vars = [v for v in variables if v in ds.data_vars]
        missing = set(variables) - set(existing_vars)
        if missing and verbose:
            print(f"Warning: Requested variables not found: {missing}")
        if not existing_vars:
            raise ValueError("No requested variables found in clipped data.")
        ds = ds[existing_vars]

    # ---- 5. Reduce 'step' dimension if present ----
    for var in ds.data_vars:
        if 'step' in ds[var].dims:
            if any(key in var.lower() for key in ['tp', 'precip', 'rain']):
                ds[var] = ds[var].sum(dim='step')
                if ds[var].attrs.get('units') == 'm':
                    ds[var] = ds[var] * 1000.0
                    ds[var].attrs['units'] = 'mm'
                if verbose:
                    print(f"Summed 'step' dimension for {var}.")
            else:
                ds[var] = ds[var].mean(dim='step')
                if verbose:
                    print(f"Averaged 'step' dimension for {var}.")

    # ---- 6. Daily resampling ----
    if resample_daily:
        daily_vars = {}
        for var in ds.data_vars:
            if any(key in var.lower() for key in ['tp', 'precip', 'rain']):
                daily = ds[var].resample(time='1D').sum()
            else:
                daily = ds[var].resample(time='1D').mean()
            daily_vars[var] = daily
        ds_daily = xr.merge(daily_vars.values())
    else:
        ds_daily = ds

    # ---- 7. Save to final NetCDF ----
    encoding = {v: {'zlib': True, 'complevel': 1, 'dtype': 'float32'}
                for v in ds_daily.data_vars}
    ds_daily.to_netcdf(output_path, encoding=encoding, compute=True)
    if verbose:
        print(f"Clipped data saved to {output_path}")

    # ---- 8. Clean up ----
    os.unlink(tmp_path)
    ds.close()
    return ds_daily

In [16]:
grib_file = "/mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib"
shapefile = "/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/dire_dawa.shp"
output_nc = "/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/climate_t2m_tp_2020.nc"

./clip_timeseries_data \
    --source=/mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib \
    --bounds=/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/dire_dawa.shp \
    --target=/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/climate_t2m_tp_2020.nc \
    --daily \
    --buffer=0.5 \
    --verbose


./clip_timeseries_data \
    --source=/mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib \
    --bounds=/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/dire_dawa.shp \
    --target=/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/climate_t2m_tp_2020.nc \
    --daily \
    --buffer=1.0 \
    --variables=2t,tp \
    --rename-t2m \
    --verbose


# Clipping (all variables)
ds = clip_grib_with_cdo(
    grib_path=grib_file,
    shapefile_path=shapefile,
    output_path=output_nc,
    buffer_deg=0.5,
    resample_daily=True,
    variables=None,   # keep all variables found
    verbose=True
)

Original bounding box (lon -180..180): [41.2280, 42.8398], lat [8.9715, 10.2837]
Converted to 0‑360: [41.2280, 42.8398]

Trying: cdo -f nc sellonlatbox 41.228027737648226,42.83977280402775,8.971483740914394,10.283741682660821 /mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib /tmp/tmp75rcroml.nc
CDO failed with stderr:

cdo (Abort): 
sellonlatbox 41.228027737648226,42.83977280402775,8.971483740914394,10.283741682660821 /mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib /tmp/tmp75rcroml.nc  
                                                                                                                                                      ^ Operator cannot be assigned.
       Reason:
         No Operators with missing input left.




CalledProcessError: Command '['cdo', '-f', 'nc', 'sellonlatbox', '41.228027737648226,42.83977280402775,8.971483740914394,10.283741682660821', '/mnt/monadworld/projects/phd/article_manuscripts/a_cdm_for_vector_of_vbds/data/datasets/climatic_timeseries_data/2020_data.grib', '/tmp/tmp75rcroml.nc']' returned non-zero exit status 1.

#### Verify data

In [ ]:
import netCDF4 as nc

with nc.Dataset(output_nc) as ds:
    print("Variables in output file:", list(ds.variables.keys()))
    print("\n" + "="*50)
    for var_name in ds.variables:
        # Skip non-data variables like time, latitude, longitude
        if var_name in ['time', 'latitude', 'longitude', 'step']:
            continue
        var = ds.variables[var_name]
        print(f"\nVariable: {var_name}")
        print(f"  Shape: {var.shape}")
        print(f"  Units: {getattr(var, 'units', 'unknown')}")
        # Print first few values (if 3D: time, lat, lon)
        if len(var.shape) == 3:
            print(f"  First 3 values (time=0, lat=0, lon=0..2): {var[0,0,:3]}")
        elif len(var.shape) == 2:
            print(f"  First 3 values (lat=0, lon=0..2): {var[0,:3]}")
        else:
            print(f"  First 3 values: {var[:3]}")

In [25]:
### Adding dummy precipitation to clipped data:

import xarray as xr
import numpy as np

# Load the existing file
ds = xr.open_dataset("/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/climate_t2m_2020.nc")

# Create a dummy tp variable with the same dimensions, filled with 0 (mm)
# Shape: (time, latitude, longitude) = (366, 6, 7)
tp_dummy = xr.DataArray(
    np.zeros((ds.dims['time'], ds.dims['latitude'], ds.dims['longitude']), dtype=np.float32),
    dims=['time', 'latitude', 'longitude'],
    coords={'time': ds.time, 'latitude': ds.latitude, 'longitude': ds.longitude},
    name='tp',
    attrs={'units': 'mm', 'long_name': 'Total precipitation (dummy)'}
)

# Add to dataset
ds['tp'] = tp_dummy

# Save to a new file (or overwrite)
output_path = "/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/climate_t2m_tp_dummy.nc"
ds.to_netcdf(output_path, encoding={'t2m': {'zlib': True, 'complevel': 1},
                                    'tp': {'zlib': True, 'complevel': 1}})
print(f"Saved with dummy tp to: {output_path}")

Saved with dummy tp to: /home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/climate_t2m_tp_dummy.nc


/tmp/ipykernel_19202/4033377607.py:12: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  np.zeros((ds.dims['time'], ds.dims['latitude'], ds.dims['longitude']), dtype=np.float32),


In [23]:
import netCDF4 as nc
# ds = nc.Dataset("/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/2020_data/2020_data_Dire_Dawa_ETDD.nc")
ds = nc.Dataset("/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/2020_data/2020_data_Dire_Dawa_ETDD.nc")
t2m = ds.variables['t2m']
print("t2m shape:", t2m.shape)
print("First 5 values at time=0, lat=0, lon=0..4:", t2m[0,0,:5])
print("Scale factor:", getattr(t2m, 'scale_factor', None))
print("Add offset:", getattr(t2m, 'add_offset', None))
print("Fill value:", getattr(t2m, '_FillValue', None))


t2m shape: (367, 2, 3)
First 5 values at time=0, lat=0, lon=0..4: [-- -- --]
Scale factor: None
Add offset: None
Fill value: nan


In [20]:

import netCDF4 as nc
ds = nc.Dataset("/home/void/Documents/codes/monadvsim/prepared_data/dire_dawa/climate_t2m_2020.nc")
t2m = ds.variables['t2m']
print("t2m shape:", t2m.shape)
print("First 5 values at time=0, lat=0, lon=0..4:", t2m[0,0,:5])
print("Scale factor:", getattr(t2m, 'scale_factor', None))
print("Add offset:", getattr(t2m, 'add_offset', None))
print("Fill value:", getattr(t2m, '_FillValue', None))


t2m shape: (366, 6, 7)
First 5 values at time=0, lat=0, lon=0..4: [300.05313 299.843   299.6572  299.28497 298.27716]
Scale factor: None
Add offset: None
Fill value: nan
